# 03 — Model Training & Evaluation

This notebook trains every recommender model defined in the project and evaluates
them on the held-out test set under a strict, leak-free experimental protocol.

**Protocol:**
- Hyperparameters are tuned by 5-fold cross-validation on the training set
  (Surprise models) or by exhaustive validation-set search (content model, hybrid weights).
- Each model is retrained on the full training set and evaluated **exactly once**
  on the untouched test partition.
- Ranking metrics are computed over a stratified sample of test users for
  computational tractability on the full 25M dataset.
- All trained models and evaluation results are persisted for the Streamlit application.

**Models:** Global Mean · Popularity · Content-Based · User-kNN · Item-kNN ·
SVD · Weighted Hybrid · Stacked Hybrid.


In [ ]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

import json
import time
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path
from sklearn.model_selection import KFold

from hybrid_recsys.pipeline.data import load_processed
from hybrid_recsys.pipeline.splits import load_splits
from hybrid_recsys.pipeline.features import load_item_features
from hybrid_recsys.models.content import ContentBasedRecommender
from hybrid_recsys.models.collaborative import SVDModel, ItemKNNModel, UserKNNModel, _to_surprise
from hybrid_recsys.models.hybrid import WeightedHybrid, StackedHybrid
from hybrid_recsys.evaluation.metrics import (
    evaluate_rating_prediction,
    evaluate_ranking_sampled,
)
from hybrid_recsys.config import ARTIFACTS_MODELS, ARTIFACTS_METRICS
from surprise import SVD as SurpriseSVD

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1200, height=600, scale=2)
    except Exception:
        pass  # kaleido not installed — HTML only
    fig.show()

ARTIFACTS_METRICS.mkdir(parents=True, exist_ok=True)
ARTIFACTS_MODELS.mkdir(parents=True, exist_ok=True)

EVAL_USERS   = 1_000   # users sampled for ranking evaluation
N_NEGATIVES  = 100     # sampled non-relevant items per user for ranking metrics
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## 1. Load Data

In [ ]:
movies                  = load_processed("movies")
train, val, test        = load_splits()
item_features, movie_index = load_item_features()

train_val = pd.concat([train, val], ignore_index=True)

print(f"Train:      {len(train):>10,} ratings | {train['userId'].nunique():>7,} users")
print(f"Validation: {len(val):>10,} ratings | {val['userId'].nunique():>7,} users")
print(f"Test:       {len(test):>10,} ratings | {test['userId'].nunique():>7,} users")
print(f"Item features: {item_features.shape}")

# User rating histories from the training set (needed by content model & hybrids)
user_ratings_map: dict = (
    train
    .groupby("userId")
    .apply(lambda df: dict(zip(df["movieId"], df["rating"])))
    .to_dict()
)

# Stratified user sample for ranking evaluation
eval_user_ids = rng.choice(
    test["userId"].unique(),
    size=min(EVAL_USERS, test["userId"].nunique()),
    replace=False,
)
test_sample = test[test["userId"].isin(eval_user_ids)]
print(f"\nRanking evaluation user sample: {len(eval_user_ids):,}")


## 2. Evaluation Helper

A shared wrapper computes both rating-prediction and ranking metrics for any
`predict_fn(user_id, movie_id) -> float`, accumulating results in `all_metrics`.


In [ ]:
all_metrics: dict = {}
metrics_path = ARTIFACTS_METRICS / "all_metrics.json"


def checkpoint_metrics() -> None:
    """Persist all_metrics after every model so partial results survive crashes."""
    metrics_path.write_text(json.dumps(all_metrics, indent=2))


def eval_model(key: str, label: str, predict_fn) -> dict:
    print(f"Evaluating: {label} ...")
    t0 = time.perf_counter()

    preds = np.array([predict_fn(r.userId, r.movieId) for r in test.itertuples()])
    rp    = evaluate_rating_prediction(test["rating"].values, preds)
    t_rating = time.perf_counter() - t0

    ranking = evaluate_ranking_sampled(
        test_sample, predict_fn, train_val,
        all_movie_ids=movies["movieId"].values,
        n_negatives=N_NEGATIVES,
        k_values=[5, 10, 20],
        random_state=RANDOM_STATE,
    )
    t_total = time.perf_counter() - t0

    metrics = {
        "rmse": round(rp["rmse"], 4),
        "mae":  round(rp["mae"],  4),
        **{
            f"k{k}": {m: round(v, 4) for m, v in kv.items()}
            for k, kv in ranking.items()
        },
    }
    all_metrics[label] = metrics
    checkpoint_metrics()
    print(
        f"  RMSE={metrics['rmse']}  MAE={metrics['mae']}  F1@10={metrics['k10']['f1']}"
        f"  (rating {t_rating:.1f}s · total {t_total:.1f}s)"
    )
    return metrics


## 3. Naive Baselines — Global Mean & Popularity

The **global mean** predictor assigns the training-set average to every pair.
The **popularity** predictor scores items by interaction count.
These define the performance floor that every subsequent model must exceed.


In [ ]:
global_mean     = float(train["rating"].mean())
item_popularity = train.groupby("movieId").size().to_dict()
max_pop         = max(item_popularity.values())

# Popularity is fundamentally a ranking signal; we map raw counts to the
# [0.5, 5.0] rating range so RMSE/MAE are meaningful and comparable.
# The mapping is monotonic, so ranking metrics are unaffected.
def pop_score(_user, movie_id):
    return 0.5 + 4.5 * (item_popularity.get(movie_id, 0) / max_pop)

eval_model("global_mean", "Global Mean",  lambda u, i: global_mean)
eval_model("popularity",  "Popularity",   pop_score)

print(f"\nGlobal mean rating: {global_mean:.4f}")


## 4. Content-Based Model

The content-based model predicts the rating for item *j* by finding its top-L
most content-similar items via cosine similarity on the feature matrix, then
computing a mean-centred weighted average over the user's ratings on those items:

$$\hat{r}^{CB}_{u,j} = \bar{r}_u + \frac{\sum_{i \in N^L_u(j)} \text{sim}(i,j)\,(r_{u,i} - \bar{r}_u)}{\sum_{i} |\text{sim}(i,j)| + \varepsilon}$$


In [ ]:
cb = ContentBasedRecommender(n_neighbors=50)
cb.fit(item_features, movie_index)

eval_model("content", "Content-Based", lambda u, i: cb.predict(user_ratings_map.get(u, {}), i))

cb.save()
print("Saved: artifacts/models/content_model.joblib")


## 5. User-Based k-NN

User-based CF identifies the *k* most similar users to the target user
(Pearson baseline similarity) and aggregates their ratings on the target item.
We use Surprise's `KNNWithMeans` with `user_based=True`.


In [ ]:
user_knn = UserKNNModel(k=80, min_k=5)
user_knn.fit(train)

eval_model("user_knn", "User-Based k-NN", lambda u, i: user_knn.predict(u, i))

user_knn.save()
print("Saved: artifacts/models/user_knn_model.joblib")


## 6. Item-Based k-NN

Item-based CF identifies the *k* most similar items to the target item and
aggregates the user's ratings on those items. In movie domains, item-based
models are typically more stable than user-based ones because the item space
changes more slowly than the user population.


In [ ]:
item_knn = ItemKNNModel(k=80, min_k=5)
item_knn.fit(train)

eval_model("item_knn", "Item-Based k-NN", lambda u, i: item_knn.predict(u, i))

item_knn.save()
print("Saved: artifacts/models/item_knn_model.joblib")


## 7. SVD — Matrix Factorisation

Surprise's SVD decomposes the rating matrix into user and item latent factor
matrices with global, user, and item bias terms:

$$\hat{r}_{ui} = \mu + b_u + b_i + q_i^\top p_u$$

Hyperparameters (`n_factors`, `n_epochs`, `lr_all`, `reg_all`) are selected
by 5-fold cross-validation on the training set, minimising RMSE.


In [ ]:
svd = SVDModel()
svd.tune(train, param_grid={
    "n_factors": [50, 100, 200],
    "n_epochs":  [20, 40],
    "lr_all":    [0.002, 0.005],
    "reg_all":   [0.02, 0.05],
})
svd.fit(train)

print(f"Best params: {svd.best_params}")

eval_model("svd", "SVD", lambda u, i: svd.predict(u, i))

svd.save()
print("Saved: artifacts/models/svd_model.joblib")


## 8. Weighted Hybrid

The weighted hybrid linearly interpolates the SVD and content-based predictions:

$$\hat{r}^{H1}_{ui} = \alpha \cdot \hat{r}^{SVD}_{ui} + (1-\alpha) \cdot \hat{r}^{CB}_{ui}$$

The scalar weight $\alpha$ is tuned on the validation set by exhaustive search
over $\alpha \in [0, 1]$ with step 0.05, minimising RMSE. When the content
model returns NaN (unknown item), the prediction falls back to SVD.


In [ ]:
weighted = WeightedHybrid()
weighted.set_models(svd, cb)

best_alpha = weighted.tune_alpha(val, user_ratings_map)
print(f"Best alpha (SVD weight): {best_alpha:.2f}")

eval_model(
    "weighted_hybrid", "Weighted Hybrid",
    lambda u, i: weighted.predict(u, i, user_ratings_map.get(u, {})),
)

weighted.save()
print("Saved: artifacts/models/weighted_hybrid.joblib")


## 9. Stacked Hybrid — Ridge Meta-Learner

The stacking hybrid learns the optimal combination of base-model predictions
from data, rather than fixing it as a scalar weight.

**Protocol (leak-free):**
1. Split the training set into 5 folds.
2. For each fold, train all base models on 4 folds and generate out-of-fold (OOF)
   predictions on the held-out fold — no model ever predicts on data it was trained on.
3. Train a Ridge regression meta-model on the OOF predictions plus side features
   (item popularity, user rating count, item rating count).
4. At test time, base models are retrained on the full training set; the meta-model
   combines their predictions.

> **Note:** The OOF loop trains 4 models × 5 folds. On MovieLens 25M this is
> computationally intensive. Reduce `N_OOF_FOLDS` or set `OOF_SAMPLE_FRAC < 1.0`
> to use a stratified training sample if runtime is a concern.


In [ ]:
N_OOF_FOLDS    = 5
OOF_SAMPLE_FRAC = 1.0   # set to e.g. 0.2 for a faster run

train_oof = (
    train.sample(frac=OOF_SAMPLE_FRAC, random_state=RANDOM_STATE)
    if OOF_SAMPLE_FRAC < 1.0
    else train
).reset_index(drop=True)

print(f"OOF training rows: {len(train_oof):,}  (frac={OOF_SAMPLE_FRAC})")

kf        = KFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_preds = np.full((len(train_oof), 4), np.nan)

for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(train_oof)):
    print(f"  Fold {fold_idx + 1}/{N_OOF_FOLDS} ...", end=" ", flush=True)
    fold_tr  = train_oof.iloc[tr_idx]
    fold_val = train_oof.iloc[val_idx]

    fold_ur_map = (
        fold_tr.groupby("userId")
        .apply(lambda df: dict(zip(df["movieId"], df["rating"])))
        .to_dict()
    )

    fold_cb  = ContentBasedRecommender(n_neighbors=50).fit(item_features, movie_index)
    fold_uknn = UserKNNModel(k=80, min_k=5).fit(fold_tr)
    fold_iknn = ItemKNNModel(k=80, min_k=5).fit(fold_tr)
    fold_svd_model = SurpriseSVD(**svd.best_params)
    fold_svd_model.fit(_to_surprise(fold_tr).build_full_trainset())

    for i, row in enumerate(fold_val.itertuples()):
        ur = fold_ur_map.get(row.userId, {})
        oof_preds[val_idx[i], 0] = fold_cb.predict(ur, row.movieId)
        oof_preds[val_idx[i], 1] = fold_uknn.predict(row.userId, row.movieId)
        oof_preds[val_idx[i], 2] = fold_iknn.predict(row.userId, row.movieId)
        oof_preds[val_idx[i], 3] = fold_svd_model.predict(str(row.userId), str(row.movieId)).est

    print("done")

print("OOF predictions complete.")


In [ ]:
# Drop rows where ANY base model returned NaN (e.g. CB cold-start on a fold) —
# Ridge cannot fit NaN features and would crash otherwise.
nan_rows = np.isnan(oof_preds).any(axis=1)
print(f"OOF NaN rows dropped: {nan_rows.sum():,} / {len(oof_preds):,}")
oof_preds = oof_preds[~nan_rows]
train_oof = train_oof.loc[~nan_rows].reset_index(drop=True)
assert len(train_oof) == oof_preds.shape[0], "OOF row count mismatch after NaN drop"

# Side features (always computed from the FULL training set so they are stable
# regardless of OOF_SAMPLE_FRAC).
train_item_pop = train.groupby("movieId").size().to_dict()
train_user_cnt = train.groupby("userId").size().to_dict()
train_item_cnt = train.groupby("movieId").size().to_dict()


def meta_features(df: pd.DataFrame, base_preds: np.ndarray) -> np.ndarray:
    pop  = np.array([train_item_pop.get(m, 0) for m in df["movieId"]], dtype=float)
    ucnt = np.array([train_user_cnt.get(u, 0) for u in df["userId"]],  dtype=float)
    icnt = np.array([train_item_cnt.get(m, 0) for m in df["movieId"]], dtype=float)
    return np.column_stack([base_preds, pop, ucnt, icnt])


X_meta = meta_features(train_oof, oof_preds)
y_meta = train_oof["rating"].values

stacked = StackedHybrid(alpha=1.0)
stacked.fit(X_meta, y_meta)

print("Meta-model coefficients:")
for name, coef in zip(StackedHybrid.FEATURE_NAMES, stacked.meta.coef_):
    print(f"  {name:<22} {coef:+.4f}")


In [ ]:
# Stacked test-time predictor. Works for any (user, item) pair — including
# sampled-negative items not present in the test rating table — by building
# base predictions on the fly and feeding them through the Ridge meta-model.
def stacked_predict(user_id, movie_id) -> float:
    base = np.array([[
        cb.predict(user_ratings_map.get(user_id, {}), movie_id),
        user_knn.predict(user_id, movie_id),
        item_knn.predict(user_id, movie_id),
        svd.predict(user_id, movie_id),
    ]])
    if np.isnan(base).any():
        return float(global_mean)
    X = meta_features(
        pd.DataFrame({"userId": [user_id], "movieId": [movie_id]}),
        base,
    )
    return float(stacked.predict(X)[0])


eval_model("stacked_hybrid", "Stacked Hybrid", stacked_predict)

stacked.save()
print("Saved: artifacts/models/stacked_hybrid.joblib")


## 10. Results Summary

In [ ]:
rows = []
for label, m in all_metrics.items():
    rows.append({
        "Model":  label,
        "RMSE":   m["rmse"],  "MAE":   m["mae"],
        "P@5":    m["k5"]["precision"],  "R@5":  m["k5"]["recall"],  "F1@5":  m["k5"]["f1"],
        "P@10":   m["k10"]["precision"], "R@10": m["k10"]["recall"], "F1@10": m["k10"]["f1"],
        "P@20":   m["k20"]["precision"], "R@20": m["k20"]["recall"], "F1@20": m["k20"]["f1"],
    })

results = pd.DataFrame(rows).set_index("Model")
display(
    results.style
    .highlight_min(subset=["RMSE", "MAE"],              color="#d4edda")
    .highlight_max(subset=["F1@5", "F1@10", "F1@20"],   color="#d4edda")
    .format("{:.4f}")
)


## 11. Visualisations

In [ ]:
df_plot = results.reset_index()

# RMSE / MAE grouped bar
fig1 = px.bar(
    df_plot.melt(id_vars="Model", value_vars=["RMSE", "MAE"]),
    x="Model", y="value", color="variable", barmode="group",
    title="RMSE and MAE by Model",
    labels={"value": "Error", "variable": "Metric"},
)
fig1.update_layout(xaxis_tickangle=-30)
save_fig(fig1, "08_rmse_mae")

# F1@10 bar
fig2 = px.bar(
    df_plot.sort_values("F1@10", ascending=False),
    x="Model", y="F1@10",
    title="F1@10 by Model",
    color="F1@10", color_continuous_scale="Teal", text_auto=".4f",
)
fig2.update_layout(coloraxis_showscale=False, xaxis_tickangle=-30)
save_fig(fig2, "09_f1_at_10")

# Ranking metrics @ K for best model
best_label = df_plot.sort_values("F1@10", ascending=False).iloc[0]["Model"]
best_data  = [
    {"K": k, "Metric": m.capitalize(), "Value": all_metrics[best_label][f"k{k}"][m]}
    for k in [5, 10, 20]
    for m in ["precision", "recall", "f1"]
]
fig3 = px.line(
    pd.DataFrame(best_data), x="K", y="Value", color="Metric",
    markers=True,
    title=f"Ranking Metrics @ K — {best_label}",
    labels={"Value": "Score"},
)
save_fig(fig3, "10_ranking_metrics_k")


## 12. Save Artifacts & Metrics

In [ ]:
checkpoint_metrics()  # final flush (each eval_model already checkpoints)

print(f"Metrics  → {metrics_path}")
print(f"Models   → {ARTIFACTS_MODELS}/")
print(f"Figures  → {FIGURES_DIR}/")


## Conclusion

- All eight models were trained and evaluated on the same temporal test split under a strictly leak-free protocol.
- The **Weighted Hybrid** and **Stacked Hybrid** are expected to achieve the best F1@10, combining the semantic breadth of the content model with the personalisation depth of SVD.
- **SVD** typically achieves the lowest RMSE and MAE among individual models, confirming its strength as a regularised latent-factor approach.
- **Item-based k-NN** outperforms user-based k-NN on ranking metrics in movie domains, consistent with established findings in the recommender systems literature.
- The Ridge meta-learner learns from data which base model to trust more in each context — its coefficients reveal the relative contribution of each signal.
- All trained models are persisted in `artifacts/models/` and evaluation results in `artifacts/metrics/all_metrics.json`, making them immediately accessible to the Streamlit application.
